#インポート

In [0]:
%pip install openpyxl -q

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import glob

#ウィジェット

In [0]:
dbutils.widgets.text("param_name", "default_value", "Parameter Description")
param_value = dbutils.widgets.get("param_name")

#読み込み

##マスター

In [0]:
# xlsxファイルの読み込み（シートごとにデータフレームを作成）
sheets = pd.read_excel("/Volumes/training_bs/common/input/master_file.xlsx",
                       dtype={"mise_cd": str, "syohin_cd": str, "syohin_cd1": str},
                       sheet_name=None)

df_sheets = {}
for name, pdf in sheets.items():
    df_sheets[name] = spark.createDataFrame(pdf)
    print(f"Sheet: {name}, Rows: {len(pdf)}, Columns: {list(pdf.columns)}")

# 各シートのデータフレームを表示
for name, df in df_sheets.items():
    print(f"\n--- {name} ---")
    display(df)

##売上データ

In [0]:
csv_files = glob.glob("/Volumes/training_bs/common/input/*.csv")
df_csvs = []
cast_types = {
    "売上店舗コード": "string",
    "レシート番号": "string",
    "商品コード": "string",
    "売上個数": "int"
}
for file in csv_files:
    df = spark.read.csv(
        file,
        header=True,
        encoding="cp932",
        inferSchema=False  
    )
    for col, dtype in cast_types.items():
        df = df.withColumn(col, F.col(col).cast(dtype))
    df_csvs.append(df)
    print(f"CSV: {file}")

if df_csvs:
    print("最初のCSVデータ")
    display(df_csvs[0])
    print("最後のCSVデータ")
    display(df_csvs[-1])